# X Space 다시듣기 저장

master_playlist.m3u8 URL로 X Space 녹음을 m4a 파일로 저장합니다. 설치할 것 없음.

**사용법**
1. 아래 **1번 셀** 왼쪽 ▶ 클릭 (준비, 최초 1회)
2. **2번 셀**에서 주의사항 5개 모두 체크 → master URL 붙여넣기 → ▶ 클릭
3. 완료되면 PC로 자동 다운로드 또는 Google Drive `XSpace` 폴더에 저장

> ⚠️ URL을 붙여넣은 상태로 이 노트북을 저장하거나 공유하지 마세요. URL에 방송 접근 키가 들어 있습니다.


In [ ]:
#@title 1. 준비 (▶ 클릭, 최초 1회)
import os, shutil, subprocess, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from urllib.error import HTTPError
from urllib.parse import urljoin
from urllib.request import Request, urlopen

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
    "Referer": "https://x.com/",
}
SEGMENT_TAGS = ("#EXTINF", "#EXT-X-PROGRAM-DATE-TIME", "#EXT-X-DISCONTINUITY")


def fetch(url, timeout=30):
    with urlopen(Request(url, headers=HEADERS), timeout=timeout) as r:
        return r.read()


def status_of(url):
    try:
        with urlopen(Request(url, headers=HEADERS), timeout=20) as r:
            r.read(1)
            return r.status
    except HTTPError as e:
        return e.code
    except OSError:
        return None


def uri_lines(text):
    return [l.strip() for l in text.splitlines() if l.strip() and not l.startswith("#")]


def dir_of(url):
    return url.split("?", 1)[0].rsplit("/", 1)[0] + "/"


def without_b(url):
    return url.replace("/b/Transcoding/", "/Transcoding/")


def base_label(b):
    k = "non_transcode" if "/non_transcode/" in b else "transcode"
    return k + (" (/b/)" if "/b/Transcoding/" in b else "")


def load_media_playlist(master_url):
    """master playlist면 첫 variant를 따라감. (playlist URL, 내용) 반환."""
    text = fetch(master_url).decode("utf-8", "replace")
    if "#EXT-X-STREAM-INF" not in text:
        return master_url, text
    variants = uri_lines(text)
    if not variants:
        raise RuntimeError("master playlist에 variant 항목 없음")
    url = urljoin(master_url, variants[0])
    last = None
    for cand in dict.fromkeys([url, without_b(url)]):
        try:
            return cand, fetch(cand).decode("utf-8", "replace")
        except OSError as e:
            last = e
    raise RuntimeError(f"variant playlist 로드 실패: {last}")


def parse_segments(text):
    header, segments, pending = [], [], []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith("#"):
            if segments or pending or line.startswith(SEGMENT_TAGS):
                pending.append(line)
            else:
                header.append(line)
        else:
            segments.append((pending, line))
            pending = []
    return header, segments, pending


def find_chunk_base(master_url, playlist_url, first_uri):
    cands = []
    for b in (dir_of(playlist_url), dir_of(master_url)):
        cands += [b, without_b(b)]
    for b in dict.fromkeys(cands):
        code = status_of(urljoin(b, first_uri))
        print(f"  경로 확인 {code}: {base_label(b)}")
        if code == 200:
            return b
    raise RuntimeError("조각 파일 경로를 찾지 못함. 녹음 삭제·비공개 전환 또는 진행 중인 Space")


def download(url, path, retries=5):
    for attempt in range(retries):
        try:
            data = fetch(url)
            with open(path, "wb") as f:
                f.write(data)
            return
        except HTTPError as e:
            if e.code in (403, 404) or attempt == retries - 1:
                raise
        except OSError:
            if attempt == retries - 1:
                raise
        time.sleep(1.5 * (attempt + 1))


def download_all(base, segments, work_dir, workers=8):
    total, failed, done = len(segments), set(), 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(download, urljoin(base, uri), os.path.join(work_dir, f"{i:06d}.aac")): i
            for i, (_, uri) in enumerate(segments)
        }
        for fut in as_completed(futures):
            i = futures[fut]
            try:
                fut.result()
            except Exception as e:  # 개별 조각 실패는 건너뜀
                failed.add(i)
                print(f"\n  조각 {i} 실패, 건너뜀: {e}")
            done += 1
            if done % 20 == 0 or done == total:
                print(f"\r  다운로드 {done}/{total}", end="", flush=True)
    print()
    return failed


def write_local_playlist(path, header, segments, tail, failed):
    lines = list(header)
    for i, (tags, _) in enumerate(segments):
        if i in failed:
            continue
        lines += tags
        lines.append(f"{i:06d}.aac")
    lines += tail
    if "#EXT-X-ENDLIST" not in lines:
        lines.append("#EXT-X-ENDLIST")
    with open(path, "w", encoding="ascii") as f:
        f.write("\n".join(lines) + "\n")


def run_ffmpeg(playlist_path, out_path):
    base = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-allowed_extensions", "ALL"]
    tail = ["-i", playlist_path, "-c", "copy", out_path]
    r = subprocess.run(base + ["-extension_picky", "0"] + tail, capture_output=True, text=True)
    if r.returncode != 0 and "extension_picky" in (r.stderr or ""):
        r = subprocess.run(base + tail, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
    return r.returncode


def unique_path(path):
    if not os.path.exists(path):
        return path
    stem, ext = os.path.splitext(path)
    n = 1
    while os.path.exists(f"{stem}_{n}{ext}"):
        n += 1
    return f"{stem}_{n}{ext}"


def save_space(master_url, out_path):
    print("[1/4] 플레이리스트 확인")
    playlist_url, text = load_media_playlist(master_url)
    header, segments, tail = parse_segments(text)
    if not segments:
        raise RuntimeError("조각 목록이 비어 있음 (진행 중인 Space이거나 녹음 없음)")
    sec = sum(float(t.split(":", 1)[1].split(",")[0]) for tags, _ in segments for t in tags if t.startswith("#EXTINF"))
    print(f"  조각 {len(segments)}개, 약 {int(sec // 60)}분 {int(sec % 60)}초")

    print("[2/4] 조각 경로 탐색")
    base = find_chunk_base(master_url, playlist_url, segments[0][1])

    work_dir = tempfile.mkdtemp(prefix="space_")
    try:
        print("[3/4] 조각 다운로드")
        failed = download_all(base, segments, work_dir)
        if len(failed) == len(segments):
            raise RuntimeError("모든 조각 다운로드 실패")

        print("[4/4] m4a로 합치기")
        local_pl = os.path.join(work_dir, "local.m3u8")
        write_local_playlist(local_pl, header, segments, tail, failed)
        if run_ffmpeg(local_pl, out_path) != 0:
            raise RuntimeError("ffmpeg 변환 실패")
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

    if failed:
        print(f"  누락된 조각 {len(failed)}개 (해당 구간 끊김)")
    return out_path


assert shutil.which("ffmpeg"), "ffmpeg 없음. 런타임 > 런타임 연결 해제 및 삭제 후 다시 실행"
print("준비 완료. 2번 셀로 이동")


In [ ]:
#@title 2. 주의사항 확인 후 저장 (모두 체크 → URL 입력 → ▶ 클릭)
#@markdown ### 주의사항 (모두 체크해야 실행됩니다)
#@markdown 다시 듣기(녹음)가 켜진, 이미 끝난 Space만 저장할 수 있습니다. 진행 중인 Space는 저장되지 않습니다.
확인1_끝난_Space만_저장_가능 = False #@param {type:"boolean"}
#@markdown 녹음의 저작권은 호스트와 발언자에게 있습니다. 개인 보관용으로만 사용하고, 재배포·공개 업로드는 허락을 받은 경우에만 합니다.
확인2_개인_보관용으로만_사용 = False #@param {type:"boolean"}
#@markdown master URL에는 방송 접근 키가 들어 있습니다. URL을 넣은 채로 이 노트북을 저장·공유하지 않습니다.
확인3_URL_공유하지_않음 = False #@param {type:"boolean"}
#@markdown 저장 중인 파일은 Google Colab 서버에 잠시 보관되며, 런타임이 끝나면 삭제됩니다.
확인4_Colab_서버_경유_이해 = False #@param {type:"boolean"}
#@markdown X의 구조가 바뀌면 동작하지 않을 수 있습니다. 사용과 결과물에 대한 책임은 사용자에게 있습니다.
확인5_사용자_책임_동의 = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 저장 설정
master_url = "" #@param {type:"string"}
파일_이름 = "" #@param {type:"string"}
#@markdown 비워두면 `space_날짜_시간.m4a`
저장_위치 = "내 PC로 다운로드" #@param ["내 PC로 다운로드", "Google Drive (XSpace 폴더)"]

checks = [확인1_끝난_Space만_저장_가능, 확인2_개인_보관용으로만_사용, 확인3_URL_공유하지_않음,
          확인4_Colab_서버_경유_이해, 확인5_사용자_책임_동의]

try:
    save_space
except NameError:
    raise SystemExit("1번 셀을 먼저 실행하세요.")

url = master_url.strip().strip('"')
name = 파일_이름.strip() or datetime.now().strftime("space_%Y%m%d_%H%M%S.m4a")
if not name.lower().endswith(".m4a"):
    name += ".m4a"

if not all(checks):
    print(f"주의사항 {len(checks) - sum(checks)}개가 체크되지 않았습니다. 모두 체크한 뒤 다시 실행하세요.")
elif not url.startswith(("http://", "https://")):
    print("master URL을 입력하세요. (https:// 로 시작)")
else:
    try:
        out_dir = "/content/output"
        if 저장_위치.startswith("Google Drive"):
            from google.colab import drive
            drive.mount("/content/drive")
            out_dir = "/content/drive/MyDrive/XSpace"
        os.makedirs(out_dir, exist_ok=True)
        out = save_space(url, unique_path(os.path.join(out_dir, name)))
        print(f"완료: {out}")
        if 저장_위치.startswith("내 PC"):
            from google.colab import files
            print("다운로드 시작 (브라우저 다운로드 목록 확인)")
            files.download(out)
    except RuntimeError as e:
        print(f"오류: {e}")
